# Stored execution evidence — Aerial Cactus Identification

This public evidence copy preserves every textual output cell supplied in `agent_final_submission.zip`.
The original notebook SHA-256 is `4d4b45d2c5f235540bf55254f6e2888149b40330f0cdc9c33f30650b95196348`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Integration Update on Colab

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `main` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on Colab.

The active flow is:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the HuggingFace dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated project runs the built-in smoke checks.


## Before running

1. In Colab, open **Secrets** and add `OPENAI_API_KEY`.
2. Add `KAGGLE_API_TOKEN` in Colab Secrets and accept the Plant Pathology competition rules on Kaggle before downloading.
3. Select a GPU runtime for real training.
4. Run the cells in order.


In [3]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path


def normalize_repo_url(value: str) -> str:
    value = (value or "").strip()

    markdown_match = re.fullmatch(
        r"\[(.*?)\]\((https?://[^)]+)\)",
        value,
    )
    if markdown_match:
        return markdown_match.group(2)

    plain_url_match = re.search(r"https?://\S+", value)
    if plain_url_match:
        return plain_url_match.group(0)

    return value


REPO_URL = normalize_repo_url(
    os.getenv(
        "JIAOZI_REPO_URL",
        "https://github.com/Isso-W/Jiaozi.git",
    )
)
REPO_REF = os.getenv("JIAOZI_REPO_REF", "main")
REPO_DIR = Path("/content/Jiaozi")

# Important:
# Colab may still be inside /content/Jiaozi from a previous run.
# Move out before deleting and cloning the repository again.
os.chdir("/content")

if REPO_DIR.exists():
    print("Removing existing repository:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)

clone_cmd = [
    "git",
    "clone",
    "--depth",
    "1",
    "--branch",
    REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

completed = subprocess.run(
    clone_cmd,
    cwd="/content",
    capture_output=True,
    text=True,
)

print("=== git stdout ===")
print(completed.stdout)

print("=== git stderr ===")
print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        "Git clone failed.\n"
        f"Return code: {completed.returncode}\n"
        f"stderr:\n{completed.stderr}"
    )

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

requirements_path = REPO_DIR / "requirements.txt"

if not requirements_path.exists():
    raise FileNotFoundError(
        f"requirements.txt was not found at {requirements_path}"
    )

print("\n=== requirements.txt ===")
print(requirements_path.read_text(encoding="utf-8")[:4000])

print("\n=== installing requirements ===")

pip_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(requirements_path),
    ],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
)

print("=== pip stdout ===")
print(pip_result.stdout)

print("=== pip stderr ===")
print(pip_result.stderr)

print("pip return code =", pip_result.returncode)

if pip_result.returncode != 0:
    raise RuntimeError(
        "pip install failed. Check the pip output above."
    )

print("Jiaozi repository:", REPO_DIR)
print("Jiaozi repo URL:", REPO_URL)
print("Jiaozi ref:", REPO_REF)

pipeline_candidates = list(REPO_DIR.rglob("pipeline.py"))

print("pipeline.py candidates:")
for path in pipeline_candidates:
    print(" -", path)

if not pipeline_candidates:
    raise FileNotFoundError(
        "No pipeline.py was found after cloning the repository."
    )


REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...


=== requirements.txt ===
# Runtime dependencies for the Jiaozi CV Auto-DL pipeline.
# Install with the same Python interpreter used to run tests, for example:
#   /usr/bin/python3 -m pip install -r requirements.txt

chromadb
datasets
networkx
numpy
openai>=2.0.0
pandas
pillow
scikit-learn
sentence-transformers
torch
torchvision
transformers

# Test runner for the module4_agent pytest suite.
pytest


=== installing requirements ===
=== pip stdout ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 145.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 111.0 MB/s eta 0

In [4]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-5.5')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token


## Download the Plant Pathology data

These cells configure the Kaggle CLI, download `plant-pathology-2020-fgvc7`, and extract the competition files under `/content/data`. The pipeline cell later converts the one-hot `train.csv` into a single `label` column that Jiaozi's local CSV loader can consume.


In [5]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

# Upgrade the packaging tools first
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

# Install the Kaggle CLI without -q so failures remain visible
kaggle_install = run_pip(["install", "--upgrade", "kaggle"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

# Read Kaggle access token from Colab Secrets
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# Verify that the Kaggle CLI is available
subprocess.run(["kaggle", "--version"], check=False)


Running: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
=== pip stdout ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.

return code = 0
Running: /usr/bin/python3 -m pip install --upgrade kaggle
=== pip stdout

CompletedProcess(args=['kaggle', '--version'], returncode=0)

In [6]:
# Configure Kaggle and download Aerial Cactus Identification dataset

KAGGLE_COMPETITION = "aerial-cactus-identification"

!mkdir -p /content/data/aerial-cactus

!kaggle competitions download \
    -c {KAGGLE_COMPETITION} \
    -p /content/data/aerial-cactus \
    --force


100% 12.0M/12.0M [00:01<00:00, 8.27MB/s]



In [7]:
# Extract Aerial Cactus competition data

from pathlib import Path
import shutil
import zipfile

DATA_ROOT = Path("/content/data/aerial-cactus")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_files = (
    sorted(DATA_ROOT.glob(f"{KAGGLE_COMPETITION}*.zip"))
    or sorted(DATA_ROOT.glob("*.zip"))
)

print("Zip files:", zip_files)

if not zip_files:
    raise FileNotFoundError(
        "No .zip file found in /content/data/aerial-cactus"
    )

zip_path = zip_files[0]

KAGGLE_DATA_DIR = (
    DATA_ROOT / "aerial-cactus-identification"
)

# Avoid mixing files from an incomplete previous extraction
if KAGGLE_DATA_DIR.exists():
    shutil.rmtree(KAGGLE_DATA_DIR)

KAGGLE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Extracting:", zip_path)
print("To:", KAGGLE_DATA_DIR)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(KAGGLE_DATA_DIR)

# Extract nested train.zip and test.zip if present
for nested_zip in sorted(
    KAGGLE_DATA_DIR.rglob("*.zip")
):
    nested_target = nested_zip.with_suffix("")

    if nested_target.exists():
        shutil.rmtree(nested_target)

    nested_target.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Extracting nested:",
        nested_zip,
        "->",
        nested_target,
    )

    with zipfile.ZipFile(
        nested_zip,
        "r",
    ) as archive:
        archive.extractall(nested_target)

print("Done.")
print("KAGGLE_DATA_DIR =", KAGGLE_DATA_DIR)

print("Top-level files:")
for path in sorted(KAGGLE_DATA_DIR.iterdir()):
    print(" -", path.name)

train_csvs = [
    path
    for path in KAGGLE_DATA_DIR.rglob("train.csv")
    if path.is_file()
]

sample_submission_csvs = [
    path
    for path in KAGGLE_DATA_DIR.rglob(
        "sample_submission.csv"
    )
    if path.is_file()
]

image_files = [
    path
    for path in KAGGLE_DATA_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png",
    }
]

if not train_csvs:
    raise FileNotFoundError(
        "train.csv was not found after extraction."
    )

if not sample_submission_csvs:
    raise FileNotFoundError(
        "sample_submission.csv was not found after extraction."
    )

if not image_files:
    raise FileNotFoundError(
        "No images were found after extraction."
    )

print("train.csv:", train_csvs[0])
print(
    "sample_submission.csv:",
    sample_submission_csvs[0],
)
print("Total extracted images:", len(image_files))


Zip files: [PosixPath('/content/data/aerial-cactus/aerial-cactus-identification.zip')]
Extracting: /content/data/aerial-cactus/aerial-cactus-identification.zip
To: /content/data/aerial-cactus/aerial-cactus-identification
Extracting nested: /content/data/aerial-cactus/aerial-cactus-identification/test.zip -> /content/data/aerial-cactus/aerial-cactus-identification/test
Extracting nested: /content/data/aerial-cactus/aerial-cactus-identification/train.zip -> /content/data/aerial-cactus/aerial-cactus-identification/train
Done.
KAGGLE_DATA_DIR = /content/data/aerial-cactus/aerial-cactus-identification
Top-level files:
 - sample_submission.csv
 - test
 - test.zip
 - train
 - train.csv
 - train.zip
train.csv: /content/data/aerial-cactus/aerial-cactus-identification/train.csv
sample_submission.csv: /content/data/aerial-cactus/aerial-cactus-identification/sample_submission.csv
Total extracted images: 21500


## Mount Google Drive for caches and checkpoints

HuggingFace caches and training checkpoints can live on Drive so a recycled Colab runtime does not force every artifact to be downloaded or trained from scratch.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/Jiaozi/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated pipeline

This cell runs the integrated Jiaozi flow against the local Kaggle files:

1. Convert Plant Pathology's one-hot labels into a local CSV with `image_id,label`.
2. Run Module 1 from the natural-language `QUERY`.
3. Run Module 2 on sampled real images from the Kaggle folder.
4. Run Module 3 retrieval and optional recommender re-ranking.
5. Run Module 4 code generation with local CSV training metadata.

Set `REAL_TRAINING = True` to generate real-training code and train it in the next cell. With `REAL_TRAINING = False`, Module 4 keeps the generated project in smoke-test mode.


In [9]:
# Aerial Cactus Kaggle data -> Jiaozi Module 1-4

import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/Jiaozi")
PIPELINE_PATH = REPO_DIR / "pipeline.py"

if not PIPELINE_PATH.exists():
    raise FileNotFoundError(
        f"No pipeline.py found at {PIPELINE_PATH}. "
        "Please rerun the git clone cell first."
    )

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Current working directory:", Path.cwd())
print("Using pipeline:", PIPELINE_PATH)


# -------------------------------------------------
# 1. Aerial Cactus task prompt
# -------------------------------------------------

QUERY = (
    "Perform binary image classification on the Kaggle Aerial Cactus "
    "Identification dataset. The dataset contains approximately 17,500 "
    "small 32-by-32 RGB aerial images. Each image is labeled "
    "has_cactus=1 when a cactus is present and has_cactus=0 otherwise. "
    "The classes are moderately imbalanced, and the task requires "
    "distinguishing cactus vegetation from visually similar aerial "
    "backgrounds. The official evaluation metric is binary ROC-AUC, "
    "which must be maximized. The model must output continuous "
    "positive-class probabilities rather than hard class labels. "
    "Use real-data training, stratified validation, pretrained image "
    "features, and efficient GPU training."
)

REAL_TRAINING = True
EPOCHS = None
OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")


# -------------------------------------------------
# 2. Find Kaggle Aerial Cactus data
# -------------------------------------------------

DATASET_DIR = Path(KAGGLE_DATA_DIR)

raw_train_csvs = sorted(
    path
    for path in DATASET_DIR.rglob("train.csv")
    if path.is_file()
)

if not raw_train_csvs:
    raise FileNotFoundError(
        f"No train.csv found under {DATASET_DIR}"
    )

raw_train_csv = raw_train_csvs[0]
raw_frame = pd.read_csv(raw_train_csv)

print("Raw train csv:", raw_train_csv)
print("Raw columns:", raw_frame.columns.tolist())
print("Raw shape:", raw_frame.shape)

required_columns = {
    "id",
    "has_cactus",
}

if not required_columns.issubset(
    raw_frame.columns
):
    raise ValueError(
        "Expected columns 'id' and 'has_cactus', "
        f"found: {raw_frame.columns.tolist()}"
    )


# -------------------------------------------------
# 3. Convert to Jiaozi-friendly CSV
# -------------------------------------------------

processed_frame = raw_frame[
    [
        "id",
        "has_cactus",
    ]
].copy()

processed_frame = processed_frame.rename(
    columns={
        "id": "image",
        "has_cactus": "label",
    }
)

processed_frame["image"] = (
    processed_frame["image"]
    .astype(str)
    .str.strip()
)

processed_frame["label"] = (
    pd.to_numeric(
        processed_frame["label"],
        errors="raise",
    )
    .astype(int)
)

if sorted(
    processed_frame["label"]
    .unique()
    .tolist()
) != [0, 1]:
    raise ValueError(
        "Aerial Cactus labels must be 0 and 1."
    )

processed_train_csv = (
    raw_train_csv.with_name(
        "jiaozi_train.csv"
    )
)

processed_frame.to_csv(
    processed_train_csv,
    index=False,
)

print("\nProcessed train csv:", processed_train_csv)
print(processed_frame.head())

print("\nClass distribution:")
print(
    processed_frame["label"]
    .value_counts()
    .sort_index()
)


# -------------------------------------------------
# 4. Locate training image directory
# -------------------------------------------------

sample_image = str(
    processed_frame.iloc[0]["image"]
)

sample_matches = [
    path
    for path in DATASET_DIR.rglob(sample_image)
    if path.is_file()
]

if not sample_matches:
    raise FileNotFoundError(
        f"Could not locate sample image "
        f"{sample_image} under {DATASET_DIR}"
    )

candidate_image_dirs = sorted(
    {
        path.parent.resolve()
        for path in sample_matches
    }
)

sample_names = (
    processed_frame["image"]
    .head(100)
    .astype(str)
    .tolist()
)

image_dir = max(
    candidate_image_dirs,
    key=lambda directory: sum(
        (directory / image_name).is_file()
        for image_name in sample_names
    ),
)

missing_images = [
    image_name
    for image_name in processed_frame[
        "image"
    ].astype(str)
    if not (
        image_dir / image_name
    ).is_file()
]

if missing_images:
    raise FileNotFoundError(
        f"{len(missing_images)} training images "
        f"are missing. First missing: "
        f"{missing_images[:10]}"
    )

print("Image directory:", image_dir)
print(
    "Verified training images:",
    len(processed_frame),
)


# -------------------------------------------------
# 5. Local benchmark information
# -------------------------------------------------

LOCAL_DATA_INFO = {
    "benchmark": "aerial_cactus_identification",
    "competition": "aerial-cactus-identification",
    "train_csv": str(processed_train_csv),
    "image_dir": str(image_dir),
    "image_column": "image",
    "label_column": "label",
    "target_column": "label",
    "image_path_template": "{image}",
    "image_extension": "",
    "num_classes": 2,
    "metric": "roc_auc",
}

print("\nPrepared local Kaggle Aerial Cactus data:")

print(
    json.dumps(
        LOCAL_DATA_INFO,
        indent=2,
        ensure_ascii=False,
    )
)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)


# -------------------------------------------------
# 6. Stable ChromaDB directory
# -------------------------------------------------

CHROMA_DIR = Path(
    "/content/jiaozi_chroma_db"
)

CHROMA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("ChromaDB directory:", CHROMA_DIR)


# -------------------------------------------------
# 7. Import Jiaozi modules
# -------------------------------------------------

from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import (
    derive_recommended_epochs,
    merge_modules,
    run_module4_generation,
)
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import (
    build_all_task_lists,
    build_graph,
    build_vector_index,
    print_results,
    retrieve_top3_hybrid,
)


# -------------------------------------------------
# 8. Module 1
# -------------------------------------------------

print(
    "\n[Notebook] Module 1: "
    "Parsing user requirements..."
)

m1_output = module1_pipeline(QUERY)

if m1_output is None:
    raise RuntimeError(
        "Module 1 failed, so no Module 3 or "
        "Module 4 output was produced."
    )


# -------------------------------------------------
# 9. Local image dataset wrapper
# -------------------------------------------------

class LocalImageSplit:
    column_names = [
        "image",
        "label",
    ]

    def __init__(
        self,
        frame,
        info,
    ):
        from PIL import Image

        self._Image = Image

        self.labels = frame[
            info["label_column"]
        ].tolist()

        image_root = Path(
            info["image_dir"]
        )

        self.image_paths = [
            image_root / str(image_name)
            for image_name in frame[
                info["image_column"]
            ].tolist()
        ]

        missing = [
            path
            for path in self.image_paths
            if not path.is_file()
        ]

        if missing:
            raise FileNotFoundError(
                f"Missing images: {missing[:10]}"
            )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, key):
        if key == "label":
            return self.labels

        if key == "image":
            return [
                self._open_image(path)
                for path in self.image_paths
            ]

        index = int(key)

        return {
            "image": self._open_image(
                self.image_paths[index]
            ),
            "label": self.labels[index],
        }

    def _open_image(self, path):
        with self._Image.open(path) as image:
            return image.convert("RGB")


def build_local_module2_dataset(info):
    frame = pd.read_csv(
        info["train_csv"]
    )

    # Module 2 only needs a representative sample.
    sample_size = min(
        600,
        len(frame),
    )

    sampled_parts = []

    for _, group in frame.groupby(
        info["label_column"]
    ):
        group_size = max(
            1,
            round(
                sample_size
                * len(group)
                / len(frame)
            ),
        )

        sampled_parts.append(
            group.sample(
                n=min(
                    group_size,
                    len(group),
                ),
                random_state=42,
            )
        )

    sample_frame = (
        pd.concat(
            sampled_parts,
            ignore_index=True,
        )
        .sample(
            frac=1,
            random_state=42,
        )
        .head(sample_size)
        .reset_index(drop=True)
    )

    print(
        "Module 2 sample:",
        f"{len(sample_frame)}/{len(frame)}",
    )

    return {
        "train": LocalImageSplit(
            sample_frame,
            info,
        )
    }


# -------------------------------------------------
# 10. Module 2
# -------------------------------------------------

print(
    "\n[Notebook] Module 2: "
    "Analyzing sampled real Kaggle images..."
)

m2_report = ImageStatisticsAnalyzer().analyze(
    build_local_module2_dataset(
        LOCAL_DATA_INFO
    )
)

# Restore real dataset statistics after sampling.
m2_report["total_images"] = int(
    len(processed_frame)
)

m2_report["num_classes"] = 2

m2_report["class_distribution"] = {
    str(label): int(count)
    for label, count in (
        processed_frame["label"]
        .value_counts()
        .sort_index()
        .to_dict()
        .items()
    )
}


# -------------------------------------------------
# 11. Merge Module 1 and Module 2
# -------------------------------------------------

m3_input = merge_modules(
    m1_output,
    m2_report,
)

m3_input["evaluation_metric"] = "roc_auc"
m3_input["num_classes"] = 2

print(
    f"\n[Notebook] Merged: "
    f"task={m3_input.get('task_type')} "
    f"size={m3_input.get('data_size')} "
    f"classes={m3_input.get('num_classes')} "
    f"metric={m3_input.get('evaluation_metric')}"
)


# -------------------------------------------------
# 12. Module 3
# -------------------------------------------------

print(
    "\n[Notebook] Module 3: "
    "Retrieving model configurations..."
)

graph = build_graph()

collection = build_vector_index(
    persist_path=str(CHROMA_DIR)
)

recommendations = retrieve_top3_hybrid(
    m3_input,
    graph,
    collection,
)

if not recommendations:
    raise RuntimeError(
        "Module 3 returned no recommendations."
    )

print_results(
    m3_input,
    recommendations,
    graph,
)

try:
    recommendations = recommend(
        recommendations,
        m2_report,
        m3_input,
        memory=OutcomeMemory(),
    )

    print(
        "[Notebook] Recommender "
        "re-ranked candidates."
    )

except Exception as exc:
    print(
        f"[Notebook] Recommender skipped: {exc}"
    )


# -------------------------------------------------
# 13. Build task lists and inject settings
# -------------------------------------------------

task_lists = build_all_task_lists(
    recommendations,
    graph,
    fmt="nl",
)

if not task_lists:
    raise RuntimeError(
        "Module 3 returned no task lists."
    )

for task_list in task_lists:
    model_config = task_list.get(
        "model_config"
    )

    if not isinstance(
        model_config,
        dict,
    ):
        continue

    model_config.update(
        {
            "num_classes": 2,
            "train_csv": LOCAL_DATA_INFO[
                "train_csv"
            ],
            "image_dir": LOCAL_DATA_INFO[
                "image_dir"
            ],
            "image_column": "image",
            "label_column": "label",
            "target_column": "label",
            "image_path_template": "{image}",
            "image_extension": "",
            "evaluation_metric": "roc_auc",
            "metric": "roc_auc",
            "metric_name": "roc_auc",
            "offline_smoke": False,
            "benchmark_key": (
                "aerial_cactus_identification"
            ),
            "competition": (
                "aerial-cactus-identification"
            ),
        }
    )

    model_config.setdefault(
        "recommended_epochs",
        derive_recommended_epochs(
            m3_input.get(
                "data_size",
                "medium",
            ),
            model_config.get(
                "finetune_strategy"
            ),
            bool(
                model_config.get(
                    "pretrained_hf_id"
                )
            ),
        ),
    )

print("\nTop model config preview:")

print(
    json.dumps(
        task_lists[0]["model_config"],
        indent=2,
        ensure_ascii=False,
    )[:3000]
)


# -------------------------------------------------
# 14. Module 4
# -------------------------------------------------

print(
    "\n[Notebook] Module 4: "
    "Generating real-training project..."
)

module4 = run_module4_generation(
    task_lists,
    OUTPUT_DIR,
    skip_smoke=REAL_TRAINING,
    timeout=120,
    llm_provider="openai",
)

summary_path = (
    OUTPUT_DIR
    / "module4_summary.json"
)

if not summary_path.exists():
    summary_path.write_text(
        json.dumps(
            module4["summary"],
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

DATASET = LOCAL_DATA_INFO["train_csv"]

print("\nModule 4 summary:", summary_path)
print(
    "Generated project output dir:",
    OUTPUT_DIR,
)
print("DATASET:", DATASET)


Current working directory: /content/Jiaozi
Using pipeline: /content/Jiaozi/pipeline.py
Raw train csv: /content/data/aerial-cactus/aerial-cactus-identification/train.csv
Raw columns: ['id', 'has_cactus']
Raw shape: (17500, 2)

Processed train csv: /content/data/aerial-cactus/aerial-cactus-identification/jiaozi_train.csv
                                  image  label
0  0004be2cfeaba1c0361d39e2b000257b.jpg      1
1  000c8a36845c0208e833c79c1bffedd1.jpg      1
2  000d1e9a533f62e55c289303b072733d.jpg      1
3  0011485b40695e9138e92d0b3fb55128.jpg      1
4  0014d7a11e90b62848904c1418fc8cf2.jpg      1

Class distribution:
label
0     4364
1    13136
Name: count, dtype: int64
Image directory: /content/data/aerial-cactus/aerial-cactus-identification/train/train
Verified training images: 17500

Prepared local Kaggle Aerial Cactus data:
{
  "benchmark": "aerial_cactus_identification",
  "competition": "aerial-cactus-identification",
  "train_csv": "/content/data/aerial-cactus/aerial-cactus-ident

Loading weights:   0%|          | 0/103 [00:01<?, ?it/s]

[Index] Added 18 backbone entries and refreshed 18 total entries.

──────────────────────────────────────────────────────────────────────
Task   : classification
Input  : size=medium  priority=accuracy
Flags  : class_imbalance
Desc   : Perform binary image classification on the Kaggle Aerial Cactus Identification dataset. The dataset contains approximately 17,500 small 32-by-32 RGB aerial images. Each image is labeled has_cactus=1 when a cactus is present and has_cactus=0 otherwise. The classes are moderately imbalanced, and the task requires distinguishing cactus vegetation from visually similar aerial backgrounds. The official evaluation metric is binary ROC-AUC, which must be maximized. The model must output continuous positive-class probabilities rather than hard class labels. Use real-data training, stratified validation, pretrained image features, and efficient GPU training.
──────────────────────────────────────────────────────────────────────

Top 1  [score=1.0  struct=1.0  vec

In [10]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


{
  "candidates": [
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 1,
      "score": 1.0,
      "task_type": "classification"
    },
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 2,
      "score": 1.0,
      "task_type": "classification"
    },
    {
      "backbone": "dinov2",
      "finetune_strategy": "full",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 3,
      "score": 0.668,
      "task_type": "classification"
    },
    {
      "backbone": "swin_transformer",
      "finetune_strategy": "full",
      "loss": "focal_loss",
      "optimizer": "adamw",
      "rank": 4,
      "score": 0.559,
      "task_type": "classification"
    }
  ],
  "errors": [],
  "generated_files": [
    "README_generated.md",
    "configs.json",
    "evaluate.py",
    "ge

## Train the recommended model (real data)

This step only runs when `REAL_TRAINING = True`. It executes the generated `run.py`,
which loads the real dataset, trains the model Module 3 recommended, evaluates on the
test split, and saves checkpoints. Select a GPU runtime first for reasonable speed.


In [11]:
# Train the recommended Aerial Cactus model
# Metric: ROC-AUC, higher is better
# Save live logs, epoch metrics and representative best checkpoints

import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

if not REAL_TRAINING:
    print(
        "REAL_TRAINING is False - "
        "skipping the real training step."
    )

else:
    SAVE_CHECKPOINTS_TO_DRIVE = True

    cfg_path = OUTPUT_DIR / "configs.json"

    configs = json.loads(
        cfg_path.read_text(
            encoding="utf-8"
        )
    )

    if not configs:
        raise ValueError(
            "configs.json is empty."
        )

    cfg = configs[0]

    model_config = cfg.get(
        "model_config"
    )

    if not isinstance(
        model_config,
        dict,
    ):
        model_config = cfg

    targets = [cfg]

    if model_config is not cfg:
        targets.append(model_config)

    for target in targets:
        target["evaluation_metric"] = "roc_auc"
        target["metric"] = "roc_auc"
        target["metric_name"] = "roc_auc"
        target["num_classes"] = 2
        target["offline_smoke"] = False

    epochs = EPOCHS

    if epochs is None:
        epochs = int(
            model_config.get(
                "recommended_epochs"
            )
            or cfg.get(
                "recommended_epochs"
            )
            or 30
        )

    backbone = (
        model_config.get("backbone")
        or cfg.get("backbone")
        or "model"
    )

    dataset_used = (
        model_config.get("dataset_id")
        or cfg.get("dataset_id")
        or DATASET
    )

    if (
        SAVE_CHECKPOINTS_TO_DRIVE
        and Path(
            "/content/drive/MyDrive"
        ).exists()
    ):
        ckpt_dir = Path(
            "/content/drive/MyDrive/"
            "Jiaozi/checkpoints/"
            "aerial_cactus_roc_auc"
        )

    elif SAVE_CHECKPOINTS_TO_DRIVE:
        raise RuntimeError(
            "Google Drive is not mounted. "
            "Run the Drive mount cell first."
        )

    else:
        ckpt_dir = (
            OUTPUT_DIR
            / "checkpoints_roc_auc"
        )

    important_dir = (
        ckpt_dir / "important_epochs"
    )

    ckpt_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    important_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for target in targets:
        target["checkpoint_dir"] = str(
            ckpt_dir
        )

        # Start a new Cactus run.
        cfg["resume_checkpoint"] = str(Path(ckpt_dir) / "best_model.pt")

    cfg_path.write_text(
        json.dumps(
            configs,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("Checkpoints ->", ckpt_dir)
    print(
        "Representative checkpoints ->",
        important_dir,
    )
    print("Resume checkpoint -> disabled")
    print("evaluation_metric -> roc_auc")

    print(
        f"Training {backbone} "
        f"for {epochs} epochs "
        f"on {dataset_used} ..."
    )

    train_command = [
        sys.executable,
        "-u",
        "run.py",
        "--epochs",
        str(epochs),
    ]

    print(
        "Command:",
        " ".join(train_command),
        "(cwd:",
        OUTPUT_DIR,
        ")",
    )

    training_log_path = (
        ckpt_dir / "training.log"
    )

    metrics_path = (
        ckpt_dir / "epoch_metrics.jsonl"
    )

    # Start clean log files for this run.
    training_log_path.write_text(
        "",
        encoding="utf-8",
    )

    metrics_path.write_text(
        "",
        encoding="utf-8",
    )

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        train_command,
        cwd=str(OUTPUT_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    if process.stdout is None:
        process.kill()

        raise RuntimeError(
            "Could not capture training output."
        )

    epoch_pattern = re.compile(
        r"epoch\s+(\d+)/(\d+).*?"
        r"loss=([0-9.eE+-]+).*?"
        r"val_roc_auc=([0-9.eE+-]+).*?"
        r"val_acc=([0-9.eE+-]+).*?"
        r"lr=([0-9.eE+-]+)",
        re.IGNORECASE,
    )

    best_pattern = re.compile(
        r"Saved new best checkpoint "
        r"at epoch\s+(\d+)",
        re.IGNORECASE,
    )

    with training_log_path.open(
        "a",
        encoding="utf-8",
        buffering=1,
    ) as log_file:

        log_file.write(
            "===== RUN START "
            f"{datetime.now().isoformat()} "
            "=====\n"
        )

        for line in process.stdout:
            print(
                line,
                end="",
                flush=True,
            )

            log_file.write(line)

            epoch_match = (
                epoch_pattern.search(line)
            )

            if epoch_match:
                epoch_record = {
                    "epoch": int(
                        epoch_match.group(1)
                    ),
                    "total_epochs": int(
                        epoch_match.group(2)
                    ),
                    "train_loss": float(
                        epoch_match.group(3)
                    ),
                    "val_roc_auc": float(
                        epoch_match.group(4)
                    ),
                    "val_accuracy": float(
                        epoch_match.group(5)
                    ),
                    "learning_rate": float(
                        epoch_match.group(6)
                    ),
                    "timestamp": (
                        datetime.now()
                        .isoformat()
                    ),
                }

                with metrics_path.open(
                    "a",
                    encoding="utf-8",
                ) as metrics_file:
                    metrics_file.write(
                        json.dumps(
                            epoch_record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

            best_match = (
                best_pattern.search(line)
            )

            if best_match:
                best_epoch = int(
                    best_match.group(1)
                )

                source_checkpoint = (
                    ckpt_dir / "best_model.pt"
                )

                if source_checkpoint.is_file():
                    snapshot_path = (
                        important_dir
                        / (
                            f"best_epoch_"
                            f"{best_epoch:03d}.pt"
                        )
                    )

                    shutil.copy2(
                        source_checkpoint,
                        snapshot_path,
                    )

                    metadata_path = (
                        important_dir
                        / (
                            f"best_epoch_"
                            f"{best_epoch:03d}.json"
                        )
                    )

                    metadata_path.write_text(
                        json.dumps(
                            {
                                "epoch": best_epoch,
                                "checkpoint": str(
                                    snapshot_path
                                ),
                                "saved_at": (
                                    datetime.now()
                                    .isoformat()
                                ),
                            },
                            indent=2,
                            ensure_ascii=False,
                        ),
                        encoding="utf-8",
                    )

                    print(
                        "[pipeline] Preserved "
                        "representative checkpoint:",
                        snapshot_path,
                        flush=True,
                    )

        log_file.write(
            "===== RUN END "
            f"{datetime.now().isoformat()} "
            "=====\n"
        )

    process.stdout.close()

    return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(
            return_code,
            train_command,
        )

    print("\nReal training finished.")
    print(
        "Best checkpoint:",
        ckpt_dir / "best_model.pt",
    )
    print(
        "Representative checkpoints:",
        important_dir,
    )
    print(
        "Epoch metrics:",
        metrics_path,
    )
    print(
        "Training log:",
        training_log_path,
    )


Checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/aerial_cactus_roc_auc
Representative checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/aerial_cactus_roc_auc/important_epochs
Resume checkpoint -> disabled
evaluation_metric -> roc_auc
Training dinov3 for 20 epochs on /content/data/aerial-cactus/aerial-cactus-identification/jiaozi_train.csv ...
Command: /usr/bin/python3 -u run.py --epochs 20 (cwd: /content/jiaozi_generated_pipeline )

Loading weights: 100%|██████████| 211/211 [00:05<00:00, 36.56it/s]
[train] requested_backbone=dinov3 hf_id=facebook/dinov3-vitb16-pretrain-lvd1689m source=huggingface actual_model=facebook/dinov3-vitb16-pretrain-lvd1689m backbone_class=_HFBackbone feature_pooling=pooler_output_or_cls_token
[train] params total=85661954 trainable=39938 backbone_trainable=38400 head_trainable=1538 other_trainable=0
[train] finetune strategy=partial unfreeze_last_n_blocks=4 frozen_backbone_param_tensors=211 partial_unfrozen_param_tensors=50
[train] Resuming from

In [2]:
from pathlib import Path

print(Path("/content/Jiaozi").exists())
print(Path("/content/data").exists())


False
False


In [1]:
!nvidia-smi


Sat Jul 11 22:46:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Show the result

Reads the best checkpoint and prints its validation score **instantly** (the metric was
already computed during training — no re-evaluation). Set `FULL_EVAL = True` for a full
re-evaluation (macro-F1, params, sample count), which costs one extra pass over the eval set.


In [12]:
import json
import torch
from pathlib import Path

ckpt_dir = Path(
    "/content/drive/MyDrive/"
    "Jiaozi/checkpoints/"
    "aerial_cactus_roc_auc"
)

best_path = (
    ckpt_dir / "best_model.pt"
)

if not best_path.exists():
    raise FileNotFoundError(
        f"best_model.pt not found: {best_path}"
    )

ckpt = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False,
)

validation = (
    ckpt.get("validation") or {}
)

metric_name = str(
    validation.get(
        "metric_name",
        "",
    )
).lower()

metric_value = validation.get(
    "metric_value"
)

print(
    "=== RESULT "
    "(best Aerial Cactus checkpoint) ==="
)

print("checkpoint :", best_path)
print(
    "best_epoch :",
    ckpt.get("best_epoch"),
)

print(
    "best_metric:",
    ckpt.get("best_metric"),
    "(best validation ROC-AUC)",
)

print(
    "validation :",
    json.dumps(
        validation,
        indent=2,
        ensure_ascii=False,
    ),
)

if metric_name not in {
    "roc_auc",
    "auc",
}:
    raise RuntimeError(
        f"Expected ROC-AUC, "
        f"but received {metric_name!r}."
    )

if metric_value is None:
    raise RuntimeError(
        "validation.metric_value is missing."
    )

metric_value = float(metric_value)

if not 0.0 <= metric_value <= 1.0:
    raise RuntimeError(
        f"Invalid ROC-AUC value: {metric_value}"
    )

print(
    "\nBest validation ROC-AUC:",
    metric_value,
)

print(
    "Accuracy at the same epoch:",
    validation.get("accuracy"),
)


=== RESULT (best Aerial Cactus checkpoint) ===
checkpoint : /content/drive/MyDrive/Jiaozi/checkpoints/aerial_cactus_roc_auc/best_model.pt
best_epoch : 20
best_metric: 0.9999223849957116 (best validation ROC-AUC)
validation : {
  "metric_name": "roc_auc",
  "metric_value": 0.9999223849957116,
  "accuracy": 0.9977142810821533,
  "epoch": 20
}

Best validation ROC-AUC: 0.9999223849957116
Accuracy at the same epoch: 0.9977142810821533


## Package the trained model for a Kaggle Notebook submission

Aerial Cactus Identification is a **Kernels-only** competition. A raw CSV uploaded from Colab is rejected with HTTP 400. This section follows the two-file workflow: it packages the best checkpoint as a Kaggle Dataset asset, while the separate `Aerial_Cactus_Jiaozi_Kaggle_Final.ipynb` performs test inference inside Kaggle and writes `/kaggle/working/submission.csv`.

After a Colab disconnect, only run the packaging cell below. It reads the existing checkpoint from Google Drive and does not retrain or rerun Modules 1–4.


In [ ]:
# Create the Kaggle Dataset asset zip from the saved Drive checkpoint
# No training and no Agent/API call occur in this cell.

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive, userdata
except Exception as exc:
    raise RuntimeError("Run this packaging cell in Google Colab.") from exc

drive.mount("/content/drive")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers"],
    check=True,
)

HF_CACHE_DIR = Path("/content/drive/MyDrive/Jiaozi/hf_cache")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

from transformers import AutoConfig

MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"
CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/Jiaozi/checkpoints/"
    "aerial_cactus_roc_auc/best_model.pt"
)
ASSET_DIR = Path("/content/aerial_cactus_dinov3_assets")
ASSET_ZIP = Path("/content/aerial_cactus_dinov3_assets.zip")

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Saved checkpoint not found: {CHECKPOINT_PATH}. "
        "Mount the same Drive account used for training."
    )

if ASSET_DIR.exists():
    shutil.rmtree(ASSET_DIR)
if ASSET_ZIP.exists():
    ASSET_ZIP.unlink()

ASSET_DIR.mkdir(parents=True, exist_ok=True)

print("Copying checkpoint to local Colab storage...")
shutil.copy2(CHECKPOINT_PATH, ASSET_DIR / "best_model.pt")

hf_token = userdata.get("HF_TOKEN") or None
config = AutoConfig.from_pretrained(MODEL_ID, token=hf_token)
config.save_pretrained(ASSET_DIR)

manifest = {
    "competition": "aerial-cactus-identification",
    "model_id": MODEL_ID,
    "checkpoint_file": "best_model.pt",
    "image_size": 224,
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225],
    "output_columns": ["id", "has_cactus"],
    "checkpoint_note": (
        "The Drive checkpoint available after the quota interruption reports "
        "best_epoch=10 and validation ROC-AUC approximately 0.99989055."
    ),
}

(ASSET_DIR / "asset_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Creating asset zip (the checkpoint is about 328 MB)...")
shutil.make_archive(
    str(ASSET_ZIP.with_suffix("")),
    "zip",
    root_dir=ASSET_DIR,
)

sha256 = hashlib.sha256()
with ASSET_ZIP.open("rb") as handle:
    for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

print("\n=== KAGGLE ASSET READY ===")
print("asset folder:", ASSET_DIR)
print("asset zip:", ASSET_ZIP)
print("zip size MB:", round(ASSET_ZIP.stat().st_size / 1024**2, 2))
print("sha256:", sha256.hexdigest())
print("files:")
for path in sorted(ASSET_DIR.iterdir()):
    print(" -", path.name, round(path.stat().st_size / 1024**2, 3), "MB")


In [ ]:
# Download the large asset zip from Colab to your computer
# If the browser blocks a large automatic download, use Colab's Files panel
# and download /content/aerial_cactus_dinov3_assets.zip manually.

from pathlib import Path
from google.colab import files

ASSET_ZIP = Path("/content/aerial_cactus_dinov3_assets.zip")
if not ASSET_ZIP.is_file():
    raise FileNotFoundError("Run the asset packaging cell first.")

print("Downloading:", ASSET_ZIP)
print("Size MB:", round(ASSET_ZIP.stat().st_size / 1024**2, 2))
files.download(str(ASSET_ZIP))
